In [41]:


## IIT Mandi Hackathon — ML Momentum Strategy
#  Full pipeline: data → features → models → backtest → metrics


import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mtick
from collections import Counter
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import xgboost as xgb

import yfinance as yf


In [ ]:

# CONFIG

TICKERS    = ['AAPL','MSFT','GOOGL','AMZN','META','TSLA','JPM','V','JNJ','BRK-B']
START      = '2017-01-01'
END        = '2025-01-31'
TC         = 0.001          # 0.1% one-way transaction cost
TOP_N      = 2              # stocks to pick each week
TRAIN_END  = '2022-12-31'
TEST_START = '2023-01-01'


In [ ]:

# 1. DATA DOWNLOAD

def download_data(tickers, start, end):
    print("=" * 60)
    print("STEP 1: Downloading OHLCV data from yfinance")
    print("=" * 60)
    raw = {}
    for tk in tickers:
        try:
            df = yf.download(tk, start=start, end=end,
                             auto_adjust=True, progress=False)
            df.columns = [c[0] if isinstance(c, tuple) else c
                          for c in df.columns]
            df = df[['Open','High','Low','Close','Volume']].ffill(limit=5).dropna()
            raw[tk] = df
            print(f"  {tk:6s}  {len(df)} rows  "
                  f"{df.index[0].date()} → {df.index[-1].date()}")
        except Exception as e:
            print(f"  {tk}: ERROR – {e}")
    print(f"\n  Downloaded {len(raw)}/{len(tickers)} tickers\n")
    return raw


In [ ]:
# 2. FEATURE ENGINEERING  (32 features per stock)

def compute_features(df: pd.DataFrame) -> pd.DataFrame:
    c, v, h, lo = df['Close'], df['Volume'], df['High'], df['Low']
    feat = pd.DataFrame(index=df.index)

    # ── Momentum
    for w in [5, 10, 21, 63]:
        feat[f'ret_{w}d'] = c.pct_change(w)
    feat['mom_1w']  = c.pct_change(5)
    feat['mom_4w']  = c.pct_change(21)
    feat['mom_13w'] = c.pct_change(63)
    feat['mom_26w'] = c.pct_change(126)
    feat['mom_52w'] = c.pct_change(252)

    # ── Short-term reversal
    feat['ret_1d'] = c.pct_change(1)
    feat['ret_2d'] = c.pct_change(2)

    # ── Moving average ratios
    for w in [10, 21, 50, 200]:
        feat[f'ma{w}_ratio'] = c / c.rolling(w).mean() - 1

    # ── Realised volatility (annualised)
    log_ret = np.log(c / c.shift(1))
    for w in [5, 21, 63]:
        feat[f'vol_{w}d'] = log_ret.rolling(w).std() * np.sqrt(252)

    # ── RSI (14-day)
    delta = c.diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    feat['rsi14'] = 100 - (100 / (1 + gain / (loss + 1e-9)))

    # ── MACD
    ema12 = c.ewm(span=12).mean()
    ema26 = c.ewm(span=26).mean()
    feat['macd']        = ema12 - ema26
    feat['macd_signal'] = feat['macd'].ewm(span=9).mean()
    feat['macd_hist']   = feat['macd'] - feat['macd_signal']

    # ── Bollinger Band position
    ma20  = c.rolling(20).mean()
    std20 = c.rolling(20).std()
    feat['bb_pos'] = (c - ma20) / (2 * std20 + 1e-9)

    # ── ATR (normalised by price)
    tr = pd.concat([h - lo,
                    (h - c.shift()).abs(),
                    (lo - c.shift()).abs()], axis=1).max(axis=1)
    feat['atr14'] = tr.rolling(14).mean() / c

    # ── Volume features
    feat['vol_ratio_5d']  = v / v.rolling(5).mean()
    feat['vol_ratio_21d'] = v / v.rolling(21).mean()
    feat['vol_trend']     = v.pct_change(5)

    # ── Price-Volume Trend change
    pvt = ((c - c.shift()) / c.shift() * v).cumsum()
    feat['pvt_chg'] = pvt.pct_change(5)

    # ── Candle geometry ─
    feat['hl_ratio']  = (h - lo) / (c + 1e-9)
    feat['close_loc'] = (c - lo) / (h - lo + 1e-9)

    # ── Return skewness
    feat['ret_skew_21d'] = log_ret.rolling(21).skew()

    return feat

def build_weekly_panel(raw_data: dict, tickers: list) -> pd.DataFrame:
    print("=" * 60)
    print("STEP 2: Feature engineering + weekly panel")
    print("=" * 60)

    all_features = {}
    for tk in tickers:
        f = compute_features(raw_data[tk])
        f['Close'] = raw_data[tk]['Close']
        all_features[tk] = f

    FEATURE_COLS = [c for c in all_features[tickers[0]].columns
                    if c != 'Close']

    weekly_dfs = []
    for tk in tickers:
        w = all_features[tk].resample('W').last()
        w['fwd_ret'] = w['Close'].pct_change(1).shift(-1)
        w['target']  = (w['fwd_ret'] > 0).astype(int)
        w['ticker']  = tk
        weekly_dfs.append(w)

    panel = pd.concat(weekly_dfs).sort_index()
    panel = panel.replace([np.inf, -np.inf], np.nan)

    print(f"  Panel shape  : {panel.shape}")
    print(f"  Date range   : {panel.index.min().date()} → {panel.index.max().date()}")
    print(f"  Features     : {len(FEATURE_COLS)}\n")
    return panel, FEATURE_COLS


In [ ]:

# 3. MODEL TRAINING

def train_models(panel: pd.DataFrame, feature_cols: list):
    print("=" * 60)
    print("STEP 3: Model training  (train ≤ 2022, test ≥ 2023)")
    print("=" * 60)

    train = panel[panel.index <= TRAIN_END].dropna(subset=feature_cols+['target'])
    test  = panel[panel.index >= TEST_START].dropna(subset=feature_cols+['target'])

    X_train, y_train = train[feature_cols], train['target']
    X_test,  y_test  = test[feature_cols],  test['target']

    scaler  = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_train)
    X_te_s  = scaler.transform(X_test)

    print(f"  Train: {len(X_train):,} samples | Test: {len(X_test):,} samples\n")

    # ── Individual models ──────────────────────────────────────
    lr = LogisticRegression(C=0.1, max_iter=1000, random_state=42)

    rf = RandomForestClassifier(
        n_estimators=300, max_depth=5,
        min_samples_leaf=10, max_features='sqrt',
        random_state=42, n_jobs=-1
    )

    xgb_mdl = xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        gamma=0.1, reg_alpha=0.1,
        use_label_encoder=False, eval_metric='logloss',
        random_state=42, n_jobs=-1
    )

    # ── Soft-Voting Ensemble
    ensemble = VotingClassifier(
        estimators=[('lr', lr), ('rf', rf), ('xgb', xgb_mdl)],
        voting='soft'
    )

    for name, model in [('Logistic Regression', lr),
                         ('Random Forest',       rf),
                         ('XGBoost',             xgb_mdl),
                         ('Ensemble (Vote)',      ensemble)]:
        model.fit(X_tr_s, y_train)
        acc = accuracy_score(y_test, model.predict(X_te_s))
        auc = roc_auc_score(y_test, model.predict_proba(X_te_s)[:,1])
        print(f"  {name:28s}  Acc={acc:.4f}  ROC-AUC={auc:.4f}")

    print()
    return ensemble, scaler, lr, rf, xgb_mdl

In [ ]:

# 4. BACKTEST

def run_backtest(panel, model, scaler, feature_cols,
                 tc=TC, top_n=TOP_N, test_start=TEST_START):
    test_dates = sorted(panel[panel.index >= test_start].index.unique())
    records    = []

    for i, date in enumerate(test_dates[:-1]):
        week_panel = panel[panel.index == date]
        avail      = week_panel.dropna(subset=feature_cols)
        if len(avail) < top_n:
            continue

        # Score & rank
        X         = scaler.transform(avail[feature_cols])
        proba     = model.predict_proba(X)[:, 1]
        avail     = avail.copy()
        avail['prob'] = proba
        ranked    = avail.sort_values('prob', ascending=False)
        selected  = ranked['ticker'].tolist()[:top_n]
        probs_sel = ranked['prob'].tolist()[:top_n]

        # Realised forward returns
        actual_rets = []
        for tk in selected:
            row = week_panel[week_panel['ticker'] == tk]
            actual_rets.append(
                float(row['fwd_ret'].iloc[0]) if len(row) else 0.0
            )

        gross_ret = np.mean(actual_rets)
        net_ret   = gross_ret - (tc * 2)   # entry + exit

        records.append({
            'date':        date,
            'stock_1':     selected[0] if len(selected) > 0 else '',
            'stock_2':     selected[1] if len(selected) > 1 else '',
            'weight_1':    0.5,
            'weight_2':    0.5,
            'prob_1':      probs_sel[0] if len(probs_sel) > 0 else 0,
            'prob_2':      probs_sel[1] if len(probs_sel) > 1 else 0,
            'ret_stock_1': actual_rets[0] if len(actual_rets) > 0 else 0,
            'ret_stock_2': actual_rets[1] if len(actual_rets) > 1 else 0,
            'gross_ret':   gross_ret,
            'tc_cost':     tc * 2,
            'net_ret':     net_ret,
        })

    return pd.DataFrame(records).set_index('date')

def add_benchmark(bt, panel, tickers, test_start=TEST_START):
    test_dates = sorted(panel[panel.index >= test_start].index.unique())
    bh_records = []
    for date in test_dates[:-1]:
        week_panel = panel[panel.index == date]
        rets = []
        for tk in tickers:
            row = week_panel[week_panel['ticker'] == tk]
            if len(row) and not pd.isna(row['fwd_ret'].iloc[0]):
                rets.append(float(row['fwd_ret'].iloc[0]))
        if rets:
            bh_records.append({'date': date, 'bh_ret': np.mean(rets)})
    bh = pd.DataFrame(bh_records).set_index('date')
    return bt.join(bh, how='left')

In [ ]:

# 5. PERFORMANCE METRICS

def calc_metrics(ret_series: pd.Series, label: str = '') -> dict:
    ret_series = ret_series.dropna()
    cum        = (1 + ret_series).cumprod()
    total_ret  = cum.iloc[-1] - 1
    n_years    = len(ret_series) / 52
    ann_ret    = (1 + total_ret) ** (1 / n_years) - 1
    ann_vol    = ret_series.std() * np.sqrt(52)
    sharpe     = ann_ret / (ann_vol + 1e-9)
    roll_max   = cum.cummax()
    drawdown   = (cum - roll_max) / roll_max
    max_dd     = drawdown.min()
    hit_rate   = (ret_series > 0).mean()
    avg_weekly = ret_series.mean()
    return dict(
        label=label, cum_ret=total_ret, ann_ret=ann_ret,
        ann_vol=ann_vol, sharpe=sharpe, max_dd=max_dd,
        hit_rate=hit_rate, avg_weekly=avg_weekly,
        cum_series=cum, dd_series=drawdown
    )

def print_metrics(gross_m, net_m, bh_m):
    print("=" * 60)
    print("STEP 5: Performance Metrics")
    print("=" * 60)
    print(f"\n{'Metric':<22} {'Gross':>10} {'Net TC':>10} {'B&H EW':>10}")
    print("-" * 55)
    rows = [
        ('Cumulative Return',  'cum_ret',    '{:.1%}'),
        ('Annualised Return',  'ann_ret',    '{:.1%}'),
        ('Annualised Vol',     'ann_vol',    '{:.1%}'),
        ('Sharpe Ratio',       'sharpe',     '{:.3f}'),
        ('Max Drawdown',       'max_dd',     '{:.1%}'),
        ('Hit Rate',           'hit_rate',   '{:.1%}'),
        ('Avg Weekly Return',  'avg_weekly', '{:.3%}'),
    ]
    for name, key, fmt in rows:
        g = fmt.format(gross_m[key])
        n = fmt.format(net_m[key])
        b = fmt.format(bh_m[key])
        print(f"{name:<22} {g:>10} {n:>10} {b:>10}")
    print("=" * 60)



In [ ]:
 #6. WALK-FORWARD RETRAINING (Advanced)

def walk_forward_backtest(panel, feature_cols,
                          tc=TC, top_n=TOP_N,
                          test_start=TEST_START,
                          retrain_every_n_weeks=26):
    print("=" * 60)
    print("STEP 6: Walk-Forward Retraining (every 26 weeks)")
    print("=" * 60)
    test_dates  = sorted(panel[panel.index >= test_start].index.unique())
    model_wf    = None
    scaler_wf   = None
    records     = []

    for i, date in enumerate(test_dates[:-1]):
        if i % retrain_every_n_weeks == 0:
            train_wf = panel[panel.index <= date].dropna(
                subset=feature_cols + ['target']
            )
            if len(train_wf) < 200:
                continue
            scaler_wf = StandardScaler()
            Xw = scaler_wf.fit_transform(train_wf[feature_cols])
            model_wf = RandomForestClassifier(
                n_estimators=200, max_depth=5,
                min_samples_leaf=10, random_state=42, n_jobs=-1
            )
            model_wf.fit(Xw, train_wf['target'])
            print(f"  Retrained @ {date.date()}  "
                  f"(train size={len(train_wf):,})")

        if model_wf is None:
            continue

        week_panel = panel[panel.index == date]
        avail      = week_panel.dropna(subset=feature_cols)
        if len(avail) < top_n:
            continue

        X         = scaler_wf.transform(avail[feature_cols])
        proba     = model_wf.predict_proba(X)[:, 1]
        avail     = avail.copy()
        avail['prob'] = proba
        ranked    = avail.sort_values('prob', ascending=False)
        selected  = ranked['ticker'].tolist()[:top_n]
        probs_sel = ranked['prob'].tolist()[:top_n]

        actual_rets = []
        for tk in selected:
            row = week_panel[week_panel['ticker'] == tk]
            actual_rets.append(
                float(row['fwd_ret'].iloc[0]) if len(row) else 0.0
            )

        gross_ret = np.mean(actual_rets)
        records.append({
            'date':        date,
            'stock_1':     selected[0] if selected else '',
            'stock_2':     selected[1] if len(selected) > 1 else '',
            'gross_ret':   gross_ret,
            'tc_cost':     tc * 2,
            'net_ret':     gross_ret - tc * 2,
        })

    bt_wf = pd.DataFrame(records).set_index('date')
    wf_m  = calc_metrics(bt_wf['net_ret'], 'Walk-Forward Net')
    print(f"\n  Ann.Return={wf_m['ann_ret']:.1%}  "
          f"Sharpe={wf_m['sharpe']:.3f}  "
          f"MaxDD={wf_m['max_dd']:.1%}\n")
    return bt_wf


In [ ]:
 #7. CHARTS

def plot_dashboard(bt, gross_m, net_m, bh_m, rf_model, feature_cols):
    print("=" * 60)
    print("STEP 7: Generating charts")
    print("=" * 60)

    A1='#00D4FF'; A2='#FF6B6B'; A3='#FFD700'
    BG='#0D1117'; CARD='#161B22'; GR='#1E2A38'; TX='#E0E0E0'

    def style(ax):
        ax.set_facecolor(CARD)
        ax.tick_params(colors=TX)
        for lab in ax.get_xticklabels()+ax.get_yticklabels():
            lab.set_color(TX)
        for sp in ax.spines.values(): sp.set_color(GR)
        ax.grid(True, color=GR, linewidth=0.6, alpha=0.8)

    fig = plt.figure(figsize=(18, 14))
    fig.patch.set_facecolor(BG)
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38)

    # 1. Cumulative Returns
    ax1 = fig.add_subplot(gs[0, :2]); style(ax1)
    ax1.plot(gross_m['cum_series'].index,
             gross_m['cum_series'] - 1, color=A3, lw=2, label='Gross')
    ax1.plot(net_m['cum_series'].index,
             net_m['cum_series'] - 1,   color=A1, lw=2, label='Net of TC')
    ax1.plot(bh_m['cum_series'].index,
             bh_m['cum_series'] - 1,    color=A2, lw=1.8, ls='--', label='B&H EW')
    ax1.axhline(0, color=GR, lw=1)
    ax1.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
    ax1.set_title('Cumulative Portfolio Return (2023–2025)',
                  fontsize=12, fontweight='bold')
    ax1.legend(fontsize=8.5)
    ax1.fill_between(net_m['cum_series'].index, 0, net_m['cum_series']-1,
                     where=(net_m['cum_series']-1 >= 0), alpha=0.12, color=A1)

    # 2. Drawdown
    ax2 = fig.add_subplot(gs[1, :2]); style(ax2)
    ax2.fill_between(net_m['dd_series'].index,
                     net_m['dd_series'], 0, color=A2, alpha=0.6, label='Net DD')
    ax2.fill_between(bh_m['dd_series'].index,
                     bh_m['dd_series'],  0, color='#888', alpha=0.3, label='B&H DD')
    ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
    ax2.set_title('Drawdown Analysis', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=8.5)

    # 3. Return Distribution
    ax3 = fig.add_subplot(gs[2, :2]); style(ax3)
    bins = np.linspace(-0.12, 0.12, 40)
    ax3.hist(bt['net_ret'], bins=bins, color=A1, alpha=0.7, label='Strategy Net')
    ax3.hist(bt['bh_ret'],  bins=bins, color=A2, alpha=0.4, label='B&H')
    ax3.axvline(bt['net_ret'].mean(), color=A1, lw=2, ls='--')
    ax3.axvline(0, color='white', lw=1, alpha=0.5)
    ax3.xaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
    ax3.set_title('Weekly Return Distribution', fontsize=12, fontweight='bold')
    ax3.legend(fontsize=8.5)

    # 4. Metrics Card
    ax4 = fig.add_subplot(gs[0, 2]); ax4.set_facecolor(CARD)
    for sp in ax4.spines.values(): sp.set_visible(False)
    ax4.set_xticks([]); ax4.set_yticks([])
    ax4.set_title('Performance Metrics', fontsize=11, fontweight='bold', color=TX)
    tbl = [
        ('Cum. Return', f"{gross_m['cum_ret']:.1%}",
                        f"{net_m['cum_ret']:.1%}", f"{bh_m['cum_ret']:.1%}"),
        ('Ann. Return', f"{gross_m['ann_ret']:.1%}",
                        f"{net_m['ann_ret']:.1%}", f"{bh_m['ann_ret']:.1%}"),
        ('Ann. Vol',    f"{gross_m['ann_vol']:.1%}",
                        f"{net_m['ann_vol']:.1%}", f"{bh_m['ann_vol']:.1%}"),
        ('Sharpe',      f"{gross_m['sharpe']:.3f}",
                        f"{net_m['sharpe']:.3f}",  f"{bh_m['sharpe']:.3f}"),
        ('Max DD',      f"{gross_m['max_dd']:.1%}",
                        f"{net_m['max_dd']:.1%}",  f"{bh_m['max_dd']:.1%}"),
        ('Hit Rate',    f"{gross_m['hit_rate']:.1%}",
                        f"{net_m['hit_rate']:.1%}", f"{bh_m['hit_rate']:.1%}"),
        ('Avg Weekly',  f"{gross_m['avg_weekly']:.3%}",
                        f"{net_m['avg_weekly']:.3%}", f"{bh_m['avg_weekly']:.3%}"),
    ]
    for j, (h, c) in enumerate(
            zip(['Metric','Gross','Net TC','B&H'], [TX, A3, A1, A2])):
        ax4.text(0.05+j*0.24, 0.96, h, transform=ax4.transAxes,
                 color=c, fontsize=7.5, fontweight='bold', va='top')
    for i, (name, g, n, b) in enumerate(tbl):
        y = 0.87 - i * 0.115
        for x, val, col in [(0.05,name,'#AAA'),(0.29,g,A3),(0.54,n,A1),(0.78,b,A2)]:
            ax4.text(x, y, val, transform=ax4.transAxes,
                     color=col, fontsize=7.2, va='top')

    # 5. Stock Frequency
    ax5 = fig.add_subplot(gs[1, 2]); style(ax5)
    sel  = list(bt['stock_1']) + list(bt['stock_2'])
    cnt  = Counter(sel)
    stks = sorted(cnt, key=lambda x: cnt[x], reverse=True)
    cols_bar = [A1 if i < 3 else '#3A5A7A' for i in range(len(stks))]
    ax5.barh(stks[::-1], [cnt[s] for s in stks[::-1]], color=cols_bar[::-1])
    ax5.set_title('Stock Selection Frequency', fontsize=10, fontweight='bold')
    ax5.set_xlabel('Times Selected', color=TX)

    # 6. Rolling Sharpe
    ax6 = fig.add_subplot(gs[2, 2]); style(ax6)
    rs  = bt['net_ret'].rolling(13).apply(
        lambda x: (x.mean()*52) / (x.std()*np.sqrt(52)+1e-9), raw=True)
    ax6.plot(rs.index, rs, color=A3, lw=1.8)
    ax6.axhline(0, color=GR, lw=1)
    ax6.axhline(1, color=A2, lw=1, ls='--', alpha=0.5)
    ax6.set_title('Rolling Sharpe (13-week)', fontsize=10, fontweight='bold')
    ax6.set_ylabel('Sharpe Ratio', color=TX)

    fig.suptitle('IIT Mandi Hackathon — ML Momentum Strategy Dashboard',
                 color='white', fontsize=16, fontweight='bold', y=0.98)
    plt.savefig('dashboard.png', dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close()
    print("  Saved: dashboard.png")

    # Feature Importance
    imp  = pd.Series(rf_model.feature_importances_,
                     index=feature_cols).sort_values(ascending=False)
    top  = imp.head(20)
    fig2, ax = plt.subplots(figsize=(10, 7))
    fig2.patch.set_facecolor(BG); ax.set_facecolor(CARD)
    cols2 = [A1 if i < 5 else '#3A6A8A' for i in range(len(top))]
    ax.barh(top.index[::-1], top.values[::-1], color=cols2[::-1])
    ax.set_title('Top 20 Feature Importances (Random Forest)',
                 color=TX, fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance', color=TX); ax.tick_params(colors=TX)
    for lab in ax.get_xticklabels()+ax.get_yticklabels(): lab.set_color(TX)
    for sp in ax.spines.values(): sp.set_color(GR)
    ax.grid(axis='x', color=GR, linewidth=0.6)
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close()
    print("  Saved: feature_importance.png\n")


In [ ]:

# MAIN

if __name__ == '__main__':

    # 1. Data
    raw_data = download_data(TICKERS, START, END)

    # 2. Features + Weekly Panel
    panel, FEATURE_COLS = build_weekly_panel(raw_data, TICKERS)

    # 3. Models
    print("=" * 60)
    print("STEP 3: Training models...")
    print("=" * 60)
    ensemble, scaler, lr, rf, xgb_mdl = train_models(panel, FEATURE_COLS)

    # 4. Backtest
    print("=" * 60)
    print("STEP 4: Running backtest...")
    print("=" * 60)
    bt = run_backtest(panel, ensemble, scaler, FEATURE_COLS)
    bt = add_benchmark(bt, panel, TICKERS)
    print(f"  Backtest weeks: {len(bt)}")
    print(bt[['stock_1','stock_2','gross_ret','net_ret']].head(8).to_string())
    print()

    # 5. Metrics
    gross_m = calc_metrics(bt['gross_ret'], 'Gross')
    net_m   = calc_metrics(bt['net_ret'],   'Net TC')
    bh_m    = calc_metrics(bt['bh_ret'],    'B&H EW')
    print_metrics(gross_m, net_m, bh_m)

    # 6. Walk-Forward
    bt_wf = walk_forward_backtest(panel, FEATURE_COLS)

    # 7. Charts
    plot_dashboard(bt, gross_m, net_m, bh_m, rf, FEATURE_COLS)

    # 8. CSV Export
    print("=" * 60)
    print("STEP 8: Exporting CSV")
    print("=" * 60)
    export = bt[['stock_1','stock_2','weight_1','weight_2',
                 'prob_1','prob_2','ret_stock_1','ret_stock_2',
                 'gross_ret','tc_cost','net_ret']].copy()
    export.index.name = 'week_end_date'
    export.to_csv('weekly_predictions_portfolio.csv')
    print("  Saved: weekly_predictions_portfolio.csv")
    print(f"  Rows : {len(export)}\n")

    print("=" * 60)
    print("  ALL DONE — check dashboard.png and the CSV!")
    print("=" * 60)


STEP 1: Downloading OHLCV data from yfinance
  AAPL    2031 rows  2017-01-03 → 2025-01-30
  MSFT    2031 rows  2017-01-03 → 2025-01-30
  GOOGL   2031 rows  2017-01-03 → 2025-01-30
  AMZN    2031 rows  2017-01-03 → 2025-01-30
  META    2031 rows  2017-01-03 → 2025-01-30
  TSLA    2031 rows  2017-01-03 → 2025-01-30
  JPM     2031 rows  2017-01-03 → 2025-01-30
  V       2031 rows  2017-01-03 → 2025-01-30
  JNJ     2031 rows  2017-01-03 → 2025-01-30
  BRK-B   2031 rows  2017-01-03 → 2025-01-30

  Downloaded 10/10 tickers

STEP 2: Feature engineering + weekly panel
  Panel shape  : (4220, 35)
  Date range   : 2017-01-08 → 2025-02-02
  Features     : 31

STEP 3: Training models...
STEP 3: Model training  (train ≤ 2022, test ≥ 2023)
  Train: 2,600 samples | Test: 1,100 samples

  Logistic Regression           Acc=0.5500  ROC-AUC=0.4748
  Random Forest                 Acc=0.5673  ROC-AUC=0.4986
  XGBoost                       Acc=0.5100  ROC-AUC=0.5060
  Ensemble (Vote)               Acc=0.533